# 00 · Setup and smoke test

This notebook runs from VS Code on a Google Colab GPU runtime. It does three things:

1. Pulls this repository from GitHub into the Colab runtime.
2. Runs the full test suite, including exact parity with the original 25.7M model.
3. Measures training throughput of every ladder rung on the current GPU and turns it into a time and compute-unit estimate.

Nothing is saved on Colab that matters: the code comes from GitHub, and the results are printed into this notebook's outputs, which live in your local `.ipynb` file.



## 1. Check the GPU

In [1]:
import subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True,
).stdout)

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07



## 2. Get the code

Set `REPO_URL` to your GitHub repository. Re-running this cell pulls the latest commit.

In [2]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"  # <- edit
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Colab already ships torch, numpy and pytest; this only fills gaps.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

commit: 9d427dd


## 3. Run the tests

The parity tests load the original model's weights into the rewritten model and require identical logits.

In [3]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "tests failed - stop here and fix before benchmarking"

....................................                                     [100%]
36 passed in 8.89s




## 4. Measure throughput

Each preset trains for a few steps on random tokens at 1024-token context with bf16 autocast. If a micro-batch does not fit in memory it is halved automatically. The result includes an MFU estimate and the hours and Colab compute units the planned pretraining token budget would need on this GPU.

In [6]:
import importlib
import json

import glm_ladder.config
import glm_ladder.model
import glm_ladder.runtime
import glm_ladder.benchmark

# Reload in dependency order so a fresh `git pull` is fully picked up.
# (Reloading only benchmark would keep the old config/model modules in memory.)
for module in (glm_ladder.config, glm_ladder.model, glm_ladder.runtime, glm_ladder.benchmark):
    importlib.reload(module)
benchmark = glm_ladder.benchmark

receipt = benchmark.run_benchmark(
    ["s-50m", "m-100m", "l-200m"],
    sequence_length=1024, micro_batch=32, steps=10,
    gradient_checkpointing=False, use_compile=False,
)
receipt["commit"] = commit
print(json.dumps(receipt, indent=2, sort_keys=True))

{
  "commit": "9d427dd",
  "environment": {
    "bf16_supported": true,
    "cuda": "12.8",
    "cuda_available": true,
    "gpu": "NVIDIA L4",
    "gpu_memory_gib": 22.03,
    "torch": "2.11.0+cu128"
  },
  "kind": "throughput_benchmark",
  "results": [
    {
      "gradient_checkpointing": false,
      "mfu_estimate": 0.043,
      "micro_batch": 16,
      "parameters": {
        "active_per_token_estimate": 26686944,
        "total": 53229024
      },
      "peak_vram_gib": 12.14,
      "preset": "s-50m",
      "pretraining_estimate": {
        "colab_compute_units": 23.0,
        "hours_on_this_gpu": 13.43,
        "target_tokens": 1500000000.0
      },
      "random_token_loss": 7.823,
      "sequence_length": 1024,
      "status": "ok",
      "step_seconds": 0.5282,
      "tokens_per_second": 31020,
      "torch_compile": false
    },
    {
      "gradient_checkpointing": false,
      "mfu_estimate": 0.044,
      "micro_batch": 16,
      "parameters": {
        "active_per_token_e

Optional: the largest rung with gradient checkpointing, which trades speed (activations are recomputed) for a larger micro-batch.

In [5]:
checkpointed = benchmark.run_benchmark(
    ["l-200m"], sequence_length=1024, micro_batch=64, steps=10,
    gradient_checkpointing=True, use_compile=False,
)
checkpointed["commit"] = commit
print(json.dumps(checkpointed, indent=2, sort_keys=True))

{
  "commit": "9d427dd",
  "environment": {
    "bf16_supported": true,
    "cuda": "12.8",
    "cuda_available": true,
    "gpu": "NVIDIA L4",
    "gpu_memory_gib": 22.03,
    "torch": "2.11.0+cu128"
  },
  "kind": "throughput_benchmark",
  "results": [
    {
      "gradient_checkpointing": true,
      "mfu_estimate": 0.041,
      "micro_batch": 32,
      "parameters": {
        "active_per_token_estimate": 93177104,
        "total": 199345424
      },
      "peak_vram_gib": 15.1,
      "preset": "l-200m",
      "pretraining_estimate": {
        "colab_compute_units": 221.7,
        "hours_on_this_gpu": 129.62,
        "target_tokens": 4000000000.0
      },
      "random_token_loss": 8.351,
      "sequence_length": 1024,
      "status": "ok",
      "step_seconds": 3.8227,
      "tokens_per_second": 8572,
      "torch_compile": false
    }
  ],
  "schema_version": "1.0",
  "settings": {
    "gradient_checkpointing": true,
    "micro_batch": 64,
    "sequence_length": 1024,
    "steps":

## 5. Next step

Save this notebook in VS Code; the printed receipts are stored in its outputs. Share the JSON so the compute plan in `PLAN.md` can be updated with measured numbers instead of estimates.

In [7]:
import torch
import torch.nn.functional as F
from torch.profiler import ProfilerActivity, profile

from glm_ladder import GLMLadderModel, get_preset

config = get_preset("s-50m")
model = GLMLadderModel(config).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, fused=True)
ids = torch.randint(0, config.vocab_size, (16, 1025), device="cuda")
inputs, labels = ids[:, :-1], ids[:, 1:]


def train_step() -> None:
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        output = model(inputs)
        loss = F.cross_entropy(output.logits.float().flatten(0, 1), labels.flatten())
        loss = loss + 0.01 * output.balance_loss
    loss.backward()
    optimizer.step()


for _ in range(3):  # warm-up: kernel selection, memory allocator
    train_step()
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(3):
        train_step()
    torch.cuda.synchronize()

averages = prof.key_averages()
try:
    print(averages.table(sort_by="self_device_time_total", row_limit=25))
except Exception:  # older column name
    print(averages.table(sort_by="self_cuda_time_total", row_limit=25))

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         3.95%      63.356ms         6.88%     110.299ms      24.365us     270.776ms        18.62%     270.776ms      59.814us          4527  
                                            aten::copy_         2.50%      39.983ms        22.13%     354.614ms      85.532us     265.179ms        18.24%     265.182ms      63.961us          4146  
         

In [8]:
compiled = benchmark.run_benchmark(
    ["s-50m"], sequence_length=1024, micro_batch=16, steps=10,
    gradient_checkpointing=False, use_compile=True,
)
compiled["commit"] = commit
print(json.dumps(compiled, indent=2, sort_keys=True))

/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
W1013 14:10:18.043000 2946 torch/_inductor/utils.py:1731] [2/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/p

{
  "commit": "9d427dd",
  "environment": {
    "bf16_supported": true,
    "cuda": "12.8",
    "cuda_available": true,
    "gpu": "NVIDIA L4",
    "gpu_memory_gib": 22.03,
    "torch": "2.11.0+cu128"
  },
  "kind": "throughput_benchmark",
  "results": [
    {
      "gradient_checkpointing": false,
      "mfu_estimate": 0.061,
      "micro_batch": 16,
      "parameters": {
        "active_per_token_estimate": 26686944,
        "total": 53229024
      },
      "peak_vram_gib": 9.45,
      "preset": "s-50m",
      "pretraining_estimate": {
        "colab_compute_units": 16.2,
        "hours_on_this_gpu": 9.47,
        "target_tokens": 1500000000.0
      },
      "random_token_loss": 7.819,
      "sequence_length": 1024,
      "status": "ok",
      "step_seconds": 0.3724,
      "tokens_per_second": 44000,
      "torch_compile": true
    }
  ],
  "schema_version": "1.0",
  "settings": {
    "gradient_checkpointing": false,
    "micro_batch": 16,
    "sequence_length": 1024,
    "steps": 10